<a href="https://colab.research.google.com/github/Gulshan-heap/sign-language-decode/blob/main/Sign_language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)

!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d mariusschmidtmengin/phoenixweather2014t-3rd-attempt

Dataset URL: https://www.kaggle.com/datasets/mariusschmidtmengin/phoenixweather2014t-3rd-attempt
License(s): unknown
100% 724M/724M [00:44<00:00, 17.1MB/s]



In [ ]:
!unzip phoenixweather2014t-3rd-attempt.zip

Streaming output truncated to the last 5000 lines.
  inflating: videos_phoenix/videos/train/09December_2009_Wednesday_tagesschau-2557.mp4  
  inflating: videos_phoenix/videos/train/09December_2009_Wednesday_tagesschau-2560.mp4  
  inflating: videos_phoenix/videos/train/09December_2010_Thursday_heute-7547.mp4  
  inflating: videos_phoenix/videos/train/09December_2010_Thursday_heute-7548.mp4  
  inflating: videos_phoenix/videos/train/09December_2010_Thursday_heute-7549.mp4  
  inflating: videos_phoenix/videos/train/09December_2010_Thursday_heute-7550.mp4  
  inflating: videos_phoenix/videos/train/09December_2010_Thursday_heute-7551.mp4  
  inflating: videos_phoenix/videos/train/09December_2010_Thursday_heute-7552.mp4  
  inflating: videos_phoenix/videos/train/09December_2010_Thursday_heute-7553.mp4  
  inflating: videos_phoenix/videos/train/09December_2010_Thursday_heute-7554.mp4  
  inflating: videos_phoenix/videos/train/09December_2010_Thursday_heute-7556.mp4  
  inflating: videos_phoe

In [ ]:
import numpy as np

In [ ]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.1 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:
# Force-upgrade protobuf to the latest release to fix TensorFlow's core components
!pip install --upgrade protobuf mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.0 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.0 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.35.0 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3

In [ ]:
!wget -O holistic_landmarker.task https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task

--2026-05-28 09:10:45--  https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task
Resolving storage.googleapis.com (storage.googleapis.com)... 172.253.118.207, 64.233.170.207, 74.125.200.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.253.118.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13683609 (13M) [application/octet-stream]
Saving to: ‘holistic_landmarker.task’

holistic_landmarker 100%[===================>]  13.05M  6.16MB/s    in 2.1s    

2026-05-28 09:10:47 (6.16 MB/s) - ‘holistic_landmarker.task’ saved [13683609/13683609]



In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2

# Initialize options for the modern Holistic Landmarker
base_options = python.BaseOptions(model_asset_path='holistic_landmarker.task')
options = vision.HolisticLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False
)

# Run the landmarker context manager
with vision.HolisticLandmarker.create_from_options(options) as landmarker:
    # Load your image array (or video frame)
    # frame = cv2.imread('path_to_frame.jpg')

    # mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    # detection_result = landmarker.detect(mp_image)

    # Now you can easily tap your point indices:
    # detection_result.face_landmarks
    # detection_result.left_hand_landmarks
    # detection_result.right_hand_landmarks
    # detection_result.pose_landmarks
    print("We're ready!!!")

We're ready!!!


#1. Extracting the hands, faces and the pose

In [ ]:
def extract_keypoints(results):
    """
    Extracts positional data from the modern MediaPipe Holistic Landmarker Result
    and flattens them into a unified single-dimensional array for your sequence models.
    """
    # 1. Extract Left Hand (21 points * 3 coordinates = 63 features)
    if results.left_hand_landmarks:
        # The new API returns landmarks as a flat list for the hand
        lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks[0]]).flatten()
    else:
        lh = np.zeros(21 * 3)

    # 2. Extract Right Hand (21 points * 3 coordinates = 63 features)
    if results.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks[0]]).flatten()
    else:
        rh = np.zeros(21 * 3)

    # 3. Extract Pose/Shoulders (33 points * 3 coordinates = 99 features)
    if results.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z] for lm in results.pose_landmarks[0]]).flatten()
    else:
        pose = np.zeros(33 * 3)

    # 4. Extract Targeted Face Landmarks (468 points * 3 coordinates = 1404 features)
    if results.face_landmarks:
        face = np.array([[lm.x, lm.y, lm.z] for lm in results.face_landmarks[0]]).flatten()
    else:
        face = np.zeros(468 * 3)

    # Concatenate everything into a single timeline array for the specific frame
    return np.concatenate([pose, face, lh, rh])

In [ ]:
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
# # using standard mediapipe

# import os
# import cv2
# from mediapipe.tasks import python
# from mediapipe.tasks.python import vision

# # --- 1. SET UP PATHS ---
# BASE_VIDEO_DIR = "/content/videos_phoenix/videos"
# BASE_OUTPUT_DIR = "/content/extracted_features"
# SPLITS = ['train', 'dev', 'test']

# # --- 2. THE FIXED EXTRACTION LOGIC (Removed [0] index references) ---
# def extract_keypoints(results):
#     # 1. Left Hand (21 points * 3 coordinates = 63 features)
#     if results.left_hand_landmarks:
#         lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks]).flatten()
#     else:
#         lh = np.zeros(21 * 3)

#     # 2. Right Hand (21 points * 3 coordinates = 63 features)
#     if results.right_hand_landmarks:
#         rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks]).flatten()
#     else:
#         rh = np.zeros(21 * 3)

#     # 3. Pose/Shoulders (33 points * 3 coordinates = 99 features)
#     if results.pose_landmarks:
#         pose = np.array([[lm.x, lm.y, lm.z] for lm in results.pose_landmarks]).flatten()
#     else:
#         pose = np.zeros(33 * 3)

#     # 4. Face Landmarks (478 points * 3 coordinates = 1434 features)
#     if results.face_landmarks:
#         face = np.array([[lm.x, lm.y, lm.z] for lm in results.face_landmarks]).flatten()
#     else:
#         face = np.zeros(478 * 3)

#     # --- HARD SAFEGUARD ---
#     # If MediaPipe fluctuates counts mid-video, force-pad/clip to target sizes
#     if pose.shape[0] != 99:
#         pose = np.zeros(99)
#     if face.shape[0] != 1434:
#         # Dynamically conform face vector to exactly 1434 elements
#         padded_face = np.zeros(1434)
#         clamped_length = min(len(face), 1434)
#         padded_face[:clamped_length] = face[:clamped_length]
#         face = padded_face
#     if lh.shape[0] != 63:
#         lh = np.zeros(63)
#     if rh.shape[0] != 63:
#         rh = np.zeros(63)

#     return np.concatenate([pose, face, lh, rh])

# # --- 3. CONFIGURING THE TASK PIPELINE ---
# base_options = python.BaseOptions(model_asset_path='holistic_landmarker.task')
# options = vision.HolisticLandmarkerOptions(base_options=base_options, output_face_blendshapes=False)

# # --- 4. THE NESTED SPLIT PROCESSING LOOP ---
# with vision.HolisticLandmarker.create_from_options(options) as landmarker:

#     for split in SPLITS:
#         input_split_dir = os.path.join(BASE_VIDEO_DIR, split)
#         output_split_dir = os.path.join(BASE_OUTPUT_DIR, split)

#         if not os.path.exists(input_split_dir):
#             print(f"Skipping split '{split}' - directory not found at {input_split_dir}")
#             continue

#         os.makedirs(output_split_dir, exist_ok=True)

#         video_files = [f for f in os.listdir(input_split_dir) if f.endswith(('.mp4', '.avi', '.mov'))]
#         print(f"\n======== Processing Split: [{split.upper()}] ({len(video_files)} videos found) ========")

#         for idx, video_name in enumerate(video_files):
#             video_path = os.path.join(input_split_dir, video_name)

#             cap = cv2.VideoCapture(video_path)
#             video_sequence = []

#             while cap.isOpened():
#                 success, frame = cap.read()
#                 if not success:
#                     break

#                 rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#                 mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

#                 results = landmarker.detect(mp_image)
#                 frame_features = extract_keypoints(results)
#                 video_sequence.append(frame_features)

#             cap.release()

#             # Save the sequence matrix
#             video_sequence = np.array(video_sequence)
#             output_filename = os.path.splitext(video_name)[0] + ".npy"
#             output_filepath = os.path.join(output_split_dir, output_filename)
#             np.save(output_filepath, video_sequence)

#             print(f" -> [{idx+1}/{len(video_files)}] Saved {output_filename} | Shape: {video_sequence.shape}")

# print("\nProcessing complete! All feature sets are split and ready.")

In [ ]:
import os
import cv2
import torch
import numpy as np
import torchvision.models.video as video_models
from torchvision.models.video import R3D_18_Weights

# --- 1. SET UP PATHS ---
BASE_VIDEO_DIR = "/content/videos_phoenix/videos"
BASE_OUTPUT_DIR = "/content/extracted_features_cnn"
SPLITS = ['train', 'dev', 'test']

# --- 2. INITIALIZE PRE-TRAINED 3D-CNN ON GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} (Make sure this says CUDA for blazing speed!)")

# Load pre-trained 3D ResNet-18 weights
weights = R3D_18_Weights.DEFAULT
model = video_models.r3d_18(weights=weights)

# Strip the final fully-connected classification layer
# This changes the model from a "video classifier" to a "feature extractor"
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval() # Set to evaluation mode

# --- 3. VIDEO PROCESSING HELPER ---
def load_and_preprocess_video(video_path, target_size=(112, 112)):
    """Reads a video, resizes frames, and converts them to a PyTorch tensor."""
    cap = cv2.VideoCapture(video_path)
    frames = []

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        # Convert BGR to RGB and resize to save GPU memory
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized_frame = cv2.resize(rgb_frame, target_size)
        frames.append(resized_frame)

    cap.release()

    if len(frames) == 0:
        return None

    # Convert list to array: [Frames, Height, Width, Channels]
    video_array = np.array(frames)

    # Rearrange dimensions to PyTorch format: [Channels, Frames, Height, Width]
    video_tensor = torch.from_numpy(video_array).float()
    video_tensor = video_tensor.permute(3, 0, 1, 2)

    # Normalize pixel values to [0, 1] as expected by ImageNet weights
    video_tensor /= 255.0

    # Add batch dimension: [1, Channels, Frames, Height, Width]
    return video_tensor.unsqueeze(0)

# --- 4. THE SPLIT CORRESPONDENCE LOOP ---
for split in SPLITS:
    input_split_dir = os.path.join(BASE_VIDEO_DIR, split)
    output_split_dir = os.path.join(BASE_OUTPUT_DIR, split)

    if not os.path.exists(input_split_dir):
        continue

    os.makedirs(output_split_dir, exist_ok=True)
    video_files = [f for f in os.listdir(input_split_dir) if f.endswith(('.mp4', '.avi', '.mov'))]
    print(f"\n======== Extracting CNN Features: [{split.upper()}] ========")

    with torch.no_grad(): # Disable gradient calculations to save memory and boost speed
        for idx, video_name in enumerate(video_files):
            video_path = os.path.join(input_split_dir, video_name)

            # Load and format the video tensor
            video_tensor = load_and_preprocess_video(video_path)
            if video_tensor is None:
                continue

            # Push data to the GPU
            video_tensor = video_tensor.to(device)

            # Forward pass through the 3D-CNN
            # Raw output shape: [1, 512, 1, 1, 1] due to pooling layers
            features = feature_extractor(video_tensor)

            # Flatten to a dense 512-dimensional vector representation of the clip
            features_np = features.cpu().numpy().flatten()

            # Save features
            output_filename = os.path.splitext(video_name)[0] + ".npy"
            np.save(os.path.join(output_split_dir, output_filename), features_np)

            if (idx + 1) % 50 == 0 or (idx + 1) == len(video_files):
                print(f" -> Progress: [{idx+1}/{len(video_files)}] Processed {video_name} | Extracted Feature Shape: {features_np.shape}")

print("\nAll video features extracted via 3D-CNN successfully!")

Using device: cuda (Make sure this says CUDA for blazing speed!)
Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:00<00:00, 186MB/s]



======== Extracting CNN Features: [TRAIN] ========
 -> Progress: [50/7096] Processed 15September_2010_Wednesday_tagesschau-5208.mp4 | Extracted Feature Shape: (512,)
 -> Progress: [100/7096] Processed 02December_2011_Friday_tagesschau-8007.mp4 | Extracted Feature Shape: (512,)
 -> Progress: [150/7096] Processed 14October_2009_Wednesday_tagesschau-518.mp4 | Extracted Feature Shape: (512,)
 -> Progress: [200/7096] Processed 22March_2011_Tuesday_tagesschau-4957.mp4 | Extracted Feature Shape: (512,)
 -> Progress: [250/7096] Processed 11May_2010_Tuesday_tagesschau-1963.mp4 | Extracted Feature Shape: (512,)
 -> Progress: [300/7096] Processed 13September_2010_Monday_heute-7171.mp4 | Extracted Feature Shape: (512,)
 -> Progress: [350/7096] Processed 16December_2011_Friday_tagesschau-6538.mp4 | Extracted Feature Shape: (512,)
 -> Progress: [400/7096] Processed 06October_2011_Thursday_heute-5542.mp4 | Extracted Feature Shape: (512,)
 -> Progress: [450/7096] Processed 26September_2010_Sunday_tag

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/extracted_features_cnn /content/drive/MyDrive/

In [ ]:
!rsync -av /content/extracted_features_cnn /content/drive/MyDrive/

Streaming output truncated to the last 5000 lines.
extracted_features_cnn/train/09December_2010_Thursday_heute-7548.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7549.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7550.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7551.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7552.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7553.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7554.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7556.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7557.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7558.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7559.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7560.npy
extracted_features_cnn/train/09December_2010_Thursday_heute-7561.npy
extracted_features_cnn/train/09December_2010_Thursda

In [ ]:
# !ls /content/extracted_features_cnn

dev  test  train


In [ ]:
!ls /content/drive/MyDrive/extracted_features_cnn

dev  test  train


In [ ]:
!zip -r extracted_features_cnn.zip /content/extracted_features_cnn

Streaming output truncated to the last 5000 lines.
  adding: content/extracted_features_cnn/train/08February_2010_Monday_heute-1495.npy (deflated 10%)
  adding: content/extracted_features_cnn/train/26May_2011_Thursday_tagesschau-4797.npy (deflated 10%)
  adding: content/extracted_features_cnn/train/28September_2010_Tuesday_tagesschau-341.npy (deflated 10%)
  adding: content/extracted_features_cnn/train/15July_2010_Thursday_tagesschau-4055.npy (deflated 10%)
  adding: content/extracted_features_cnn/train/12March_2011_Saturday_tagesschau-2459.npy (deflated 10%)
  adding: content/extracted_features_cnn/train/02October_2010_Saturday_tagesschau-1301.npy (deflated 10%)
  adding: content/extracted_features_cnn/train/06December_2011_Tuesday_tagesschau-6847.npy (deflated 10%)
  adding: content/extracted_features_cnn/train/13April_2011_Wednesday_tagesschau-1676.npy (deflated 10%)
  adding: content/extracted_features_cnn/train/08April_2010_Thursday_tagesschau-3958.npy (deflated 10%)
  adding: con

In [ ]:
from google.colab import files
files.download('/content/extracted_features_cnn.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>